# ✈️ Airline Delay Analytics — Silver Layer Exploratory Data Analysis

**Project:** Airline Delay Analytics (AWS S3 Bronze → Silver → Gold Architecture)
**Layer:** Silver
**Engine:** Apache Spark (PySpark) on Amazon EMR (m4.large cluster)
**Downstream stack:** Gold Layer Feature Engineering → Machine Learning → Power BI Dashboarding

## Notebook Purpose

This notebook performs exploratory data analysis (EDA) on the Silver-layer flight dataset. It validates data quality and business rules, profiles delay and cancellation behaviour across airlines, airports, and time, verifies a composite business key, and concludes with a formal feature-selection and Data Understanding summary that will drive Gold-layer development.

## Notebook Contents

1. Environment Setup
2. Data Loading — Silver Layer
3. Column Selection for Exploratory Analysis
4. Dataset Overview
5. Missing Value Analysis — Part 1 (Raw Counts)
6. Numeric Summary Statistics
7. Categorical Value Distribution
8. Delay and Cancellation Distribution
9. Business Rule Validation — Part 1 (Delay Threshold Logic)
10. Candidate Key Utility Function
11. Missing Value Analysis — Part 2 (Percentage View)
12. Business Rule Validation — Part 2 (Null Composition by Operational Status)
13. Business Rule Validation — Part 1 Continued (Count-Based Confirmation)
14. Airline Performance Analysis
15. Airport Traffic Analysis
16. Monthly Trend Analysis
17. Correlation Analysis
18. Composite Business Key Validation — Part 1 (Five-Column Candidate)
19. Delay Cause Analysis
20. Composite Business Key Validation — Part 2 (Final Six-Column Key)
21. Business Rule Validation — Detailed Null-Composition Walkthrough
22. Note on Retained Placeholder Cell
23. Full Raw Schema Reference
24. EDA Summary
25. Data Understanding & Feature Selection

> **Note on compute strategy:** This notebook is optimized for a modest `m4.large` EMR cluster. Analysis is deliberately scoped to a focused subset of business-critical columns (out of 120 raw Silver columns) to minimize the number of expensive full-table Spark actions (`count()`, `show()`, `describe()`) triggered during exploration.

---


## 1. Environment Setup

Core PySpark SQL functions and data types are imported. These provide the column expressions (`col`, `when`, `count`, `avg`, `sum`, `desc`, ...) and type classes used throughout the notebook.


In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
2,application_1784443207008_0003,pyspark3,idle,Link,Link,✔


SparkSession available as 'spark'.


## 2. Data Loading — Silver Layer

The curated Silver-layer flight dataset — already cleaned and standardized by the upstream Bronze → Silver pipeline — is loaded directly from Amazon S3 in Parquet format. Parquet is used because it is columnar, compressed, and well suited to large-scale Spark analytics on EMR.


In [2]:
df = spark.read.parquet(
    "s3a://airline-dataset-2020-2025/Silver/Flight_Data_2020_2025/"
)

## 3. Column Selection for Exploratory Analysis

The raw Silver dataset contains 120 columns (see Section 23 for the full schema), many of which are diversion-specific, redundant, or not relevant to delay analytics. To keep this EDA efficient on a single `m4.large` node, a focused list of business-critical columns is selected up front, covering flight identifiers, schedule/time attributes, airline and airport information, delay indicators, and operational flags.


In [3]:
important_cols=[
'FlightDate','Year','Quarter','Month','DayOfMonth','DayOfWeek',
'Operating_Airline','Marketing_Airline_Network',
'Flight_Number_Operating_Airline','Tail_Number',
'Origin','OriginCityName','OriginStateName',
'Dest','DestCityName','DestStateName',
'CRSDepTime','DepTime','CRSArrTime','ArrTime',
'DepDelay','DepDel15','ArrDelay','ArrDel15',
'Cancelled','CancellationCode','Diverted',
'TaxiOut','TaxiIn','AirTime','Distance'
]
available=[c for c in important_cols if c in df.columns]
eda_df=df.select(*available)
print("Columns:",len(available))
print(available)


Columns: 30
['FlightDate', 'Year', 'Quarter', 'Month', 'DayOfWeek', 'Operating_Airline', 'Marketing_Airline_Network', 'Flight_Number_Operating_Airline', 'Tail_Number', 'Origin', 'OriginCityName', 'OriginStateName', 'Dest', 'DestCityName', 'DestStateName', 'CRSDepTime', 'DepTime', 'CRSArrTime', 'ArrTime', 'DepDelay', 'DepDel15', 'ArrDelay', 'ArrDel15', 'Cancelled', 'CancellationCode', 'Diverted', 'TaxiOut', 'TaxiIn', 'AirTime', 'Distance']

### Observation

The column-selection filter retained **30 of the requested columns**. Comparing the printed column list against the full raw schema (Section 23) shows that `DayOfMonth` is missing from the result — the raw Silver schema stores this field as `DayofMonth` (lowercase "of"), so the case-sensitive membership check `c in df.columns` silently dropped it. All other requested time, airline, airport, delay, and operational fields were retained successfully.

### Business Insight

A single case mismatch in a column name is a small issue in EDA, but it can silently remove a business-relevant feature (day-of-month, useful for identifying paycheck-cycle or holiday travel patterns) from downstream analysis without raising an error. This is flagged as a Gold-layer data-engineering action item: standardize column-naming conventions between the Bronze/Silver ingestion pipeline and the analytics layer so that case-sensitive lookups cannot silently drop fields.


## 4. Dataset Overview

With the analysis-ready column set defined, the schema and a sample of records are inspected to confirm data types and verify that the load and column selection behaved as expected.


In [4]:
eda_df.printSchema()
eda_df.show(5,truncate=False)

root
 |-- FlightDate: timestamp (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- Operating_Airline: string (nullable = true)
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- Flight_Number_Operating_Airline: integer (nullable = true)
 |-- Tail_Number: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- OriginStateName: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestCityName: string (nullable = true)
 |-- DestStateName: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- DepTime: integer (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- ArrTime: integer (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- DepDel15: double (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- ArrDel15: double (nullable = true

### Observation

The schema confirms appropriate data typing: `FlightDate` is a `timestamp`, delay- and distance-related fields (`DepDelay`, `ArrDelay`, `TaxiOut`, `TaxiIn`, `AirTime`, `Distance`) are stored as `double`, and flag fields (`Cancelled`, `Diverted`, `DepDel15`, `ArrDel15`) are stored as `double` (0.0 / 1.0) rather than boolean. The 5-row sample shows realistic, well-formed flight records (e.g., Alaska Airlines flights between ORD, PDX, LAS, and SEA) with consistent scheduled-versus-actual time values and small delay magnitudes.

### Business Insight

Flag columns being stored as `double` rather than `boolean`/`integer` is a minor schema-design inefficiency worth correcting in the Gold layer — casting to `boolean` or `tinyint` reduces storage footprint and simplifies downstream filtering logic in Spark SQL and Power BI. The sample data otherwise confirms the Silver layer is analysis-ready.


## 5. Missing Value Analysis — Part 1: Raw Counts

Null counts are computed for every selected column to establish a baseline view of data completeness before any statistical or business-rule analysis is performed.


In [5]:
missing=eda_df.select([count(when(col(c).isNull(),1)).alias(c) for c in eda_df.columns])
missing.show(truncate=False)

+----------+----+-------+-----+---------+-----------------+-------------------------+-------------------------------+-----------+------+--------------+---------------+----+------------+-------------+----------+-------+----------+-------+--------+--------+--------+--------+---------+----------------+--------+-------+------+-------+--------+
|FlightDate|Year|Quarter|Month|DayOfWeek|Operating_Airline|Marketing_Airline_Network|Flight_Number_Operating_Airline|Tail_Number|Origin|OriginCityName|OriginStateName|Dest|DestCityName|DestStateName|CRSDepTime|DepTime|CRSArrTime|ArrTime|DepDelay|DepDel15|ArrDelay|ArrDel15|Cancelled|CancellationCode|Diverted|TaxiOut|TaxiIn|AirTime|Distance|
+----------+----+-------+-----+---------+-----------------+-------------------------+-------------------------------+-----------+------+--------------+---------------+----+------------+-------------+----------+-------+----------+-------+--------+--------+--------+--------+---------+----------------+--------+-------

### Observation

`CancellationCode` has by far the largest null count (39,993,169 of 40,910,253 rows) — expected, since this field is only populated for flights that were actually cancelled. Delay- and time-related fields (`ArrDelay`, `ArrDel15`, `AirTime`, `ArrTime`, `TaxiIn`, `TaxiOut`, `DepTime`, `DepDelay`, `DepDel15`) each have roughly 895,000–1,015,000 nulls, and `Tail_Number` has 296,073 nulls. All identifier, schedule, and route columns are fully populated.

### Business Insight

The nulls are not randomly scattered — they cluster in outcome-dependent fields that are simply undefined for flights that never departed or never landed as scheduled. This raw count establishes the baseline that is examined in percentage terms in Section 11 and validated against operational status in Sections 12 and 21.


## 6. Numeric Summary Statistics

Descriptive statistics (count, mean, standard deviation, min, max) are computed for all numeric columns to understand the distribution and range of delay, timing, and distance metrics prior to feature engineering.


In [6]:
numeric=[f.name for f in eda_df.schema.fields if isinstance(f.dataType,(IntegerType,LongType,DoubleType,FloatType,ShortType,DecimalType))]
eda_df.select(numeric).describe().show()

+-------+------------------+------------------+------------------+------------------+-------------------------------+------------------+-----------------+------------------+-----------------+------------------+------------------+-----------------+------------------+-------------------+--------------------+-----------------+-----------------+------------------+-----------------+
|summary|              Year|           Quarter|             Month|         DayOfWeek|Flight_Number_Operating_Airline|        CRSDepTime|          DepTime|        CRSArrTime|          ArrTime|          DepDelay|          DepDel15|         ArrDelay|          ArrDel15|          Cancelled|            Diverted|          TaxiOut|           TaxiIn|           AirTime|         Distance|
+-------+------------------+------------------+------------------+------------------+-------------------------------+------------------+-----------------+------------------+-----------------+------------------+------------------+---------

### Observation

Average departure delay is **10.94 minutes** and average arrival delay is **5.24 minutes**, with high standard deviations (53.0 and 54.9 respectively) indicating a right-skewed distribution — most flights are on time or early, but a subset of severely delayed flights (maximum observed: 7,223 minutes departure, 7,232 minutes arrival) pulls the mean upward. The `Cancelled` column has a mean of **0.0224** and `Diverted` a mean of **0.0024**, directly indicating a **2.24% cancellation rate** and a **0.24% diversion rate** across the dataset. Average taxi-out time (17.3 min) exceeds average taxi-in time (8.0 min), consistent with typical ground-operations patterns.

### Business Insight

A ~2.2% cancellation rate combined with heavy right-skew in delay minutes is an important input for the Gold-layer reliability-scoring model — the mean delay alone understates the experience of the "long tail" of severely disrupted flights. Both mean-based and threshold-based (`ArrDel15`) features should be carried forward for reliability KPIs and predictive modelling.


## 7. Categorical Value Distribution — Top Values

To understand the composition of key categorical dimensions, the top 10 most frequent values are examined for the first five string-typed columns (operating airline, marketing airline, tail number, origin airport, and origin city).


In [7]:
cat=[f.name for f in eda_df.schema.fields if isinstance(f.dataType,StringType)]
for c in cat[:5]:
    print("="*50)
    print(c)
    eda_df.groupBy(c).count().orderBy(desc("count")).show(10,False)

Operating_Airline
+-----------------+-------+
|Operating_Airline|count  |
+-----------------+-------+
|WN               |7582834|
|DL               |5242763|
|AA               |5078840|
|OO               |4343633|
|UA               |3669461|
|YX               |1812974|
|MQ               |1521850|
|B6               |1366470|
|9E               |1354202|
|OH               |1300460|
+-----------------+-------+
only showing top 10 rows

Marketing_Airline_Network
+-------------------------+--------+
|Marketing_Airline_Network|count   |
+-------------------------+--------+
|AA                       |10407897|
|DL                       |8526129 |
|WN                       |7582834 |
|UA                       |7427635 |
|AS                       |2238407 |
|B6                       |1366470 |
|NK                       |1278352 |
|F9                       |968030  |
|G4                       |694895  |
|HA                       |419604  |
+-------------------------+--------+

Tail_Number
+------

### Observation

Southwest (`WN`) is the largest **operating** carrier with 7.58M flights, while American Airlines (`AA`) is the largest **marketing** carrier with 10.4M flights — a meaningful gap explained by codeshare and regional-partner operations (many American-marketed flights are physically operated by regional partners under `Operating_Airline`). Chicago (`ORD`) and Atlanta (`ATL`) are the busiest origin cities/airports.

### Business Insight

The gap between "marketing airline" and "operating airline" volumes confirms that codeshare relationships are material in this dataset. Any airline-level performance analysis (e.g., an on-time-performance leaderboard for the Power BI dashboard) must clearly state whether it is scored by `Operating_Airline` (who actually flew the aircraft) or `Marketing_Airline_Network` (who sold the ticket) — the two produce materially different rankings, and misattributing delay responsibility could unfairly penalize or credit the wrong carrier.


## 8. Delay and Cancellation Distribution

This step quantifies how many flights are classified as delayed (15+ minutes) versus on-time, and how many were cancelled, using the pre-computed `ArrDel15`, `DepDel15`, and `Cancelled` flag columns.


In [8]:
eda_df.groupBy("ArrDel15").count().show()
eda_df.groupBy("DepDel15").count().show()
eda_df.groupBy("Cancelled").count().show()

+--------+--------+
|ArrDel15|   count|
+--------+--------+
|     0.0|32251102|
|    null| 1014879|
|     1.0| 7644272|
+--------+--------+

+--------+--------+
|DepDel15|   count|
+--------+--------+
|     0.0|32448294|
|    null|  896518|
|     1.0| 7565441|
+--------+--------+

+---------+--------+
|Cancelled|   count|
+---------+--------+
|      0.0|39993169|
|      1.0|  917084|
+---------+--------+

### Observation

**18.7%** of flights with a recorded arrival status (7,644,272 of 40,910,253) arrived 15+ minutes late (`ArrDel15 = 1`), and **18.5%** departed 15+ minutes late (7,565,441 flights). **2.24%** of all flights (917,084) were cancelled. The null counts for `ArrDel15` (1,014,879) and `DepDel15` (896,518) match the missing-value counts identified in Section 5, reinforcing that these nulls stem from cancelled/diverted flights rather than data loss.

### Business Insight

Roughly 1 in 5 flights experiences a significant delay — a rate substantial enough to justify investing in a predictive delay model for the Gold layer and dashboard alerting. The agreement between the missing `ArrDel15`/`DepDel15` counts here and the missing-value analysis in Section 5 is itself a useful data-quality confirmation, cross-validating two independently computed views of the dataset.


## 9. Business Rule Validation — Part 1: Delay Threshold Logic

The Department of Transportation convention defines a "delayed" flight as one with 15 or more minutes of delay. This section validates that the pre-computed `ArrDel15`/`DepDel15` flags are perfectly consistent with the underlying `ArrDelay`/`DepDelay` minute values — i.e., that no record has a delay of 15+ minutes without being flagged as delayed. The row-level check below is confirmed with an explicit count in Section 13.


In [9]:
eda_df.filter((col("ArrDelay")>=15)&(col("ArrDel15")!=1)).show(20,False)
eda_df.filter((col("DepDelay")>=15)&(col("DepDel15")!=1)).show(20,False)

+----------+----+-------+-----+---------+-----------------+-------------------------+-------------------------------+-----------+------+--------------+---------------+----+------------+-------------+----------+-------+----------+-------+--------+--------+--------+--------+---------+----------------+--------+-------+------+-------+--------+
|FlightDate|Year|Quarter|Month|DayOfWeek|Operating_Airline|Marketing_Airline_Network|Flight_Number_Operating_Airline|Tail_Number|Origin|OriginCityName|OriginStateName|Dest|DestCityName|DestStateName|CRSDepTime|DepTime|CRSArrTime|ArrTime|DepDelay|DepDel15|ArrDelay|ArrDel15|Cancelled|CancellationCode|Diverted|TaxiOut|TaxiIn|AirTime|Distance|
+----------+----+-------+-----+---------+-----------------+-------------------------+-------------------------------+-----------+------+--------------+---------------+----+------------+-------------+----------+-------+----------+-------+--------+--------+--------+--------+---------+----------------+--------+-------

## 10. Candidate Key Utility Function

A reusable helper function is defined to test whether a given combination of columns uniquely identifies each row — i.e., whether it is a valid candidate key — by comparing the total row count to the count of distinct combinations of those columns. This function's logic underpins the manual key checks carried out in Sections 18 and 20.


In [10]:
def check_candidate_key(cols):
    total=eda_df.select(*cols).count()
    unique=eda_df.select(*cols).dropDuplicates().count()
    print("Rows:",total)
    print("Unique:",unique)
    print("Candidate Key" if total==unique else "Not Unique")

## 11. Missing Value Analysis — Part 2: Percentage View

Building on the raw counts from Section 5, missing values are now expressed as percentages — first as a single wide-format aggregate query, then as a ranked, per-column breakdown — to make the scale of missingness easier to interpret and communicate.

### 11.1 Aggregate Percentage View


In [11]:
rows = eda_df.count()

null_df = eda_df.select([
    (
        (count(when(col(c).isNull(), c)) / rows) * 100
    ).alias(c)
    for c in eda_df.columns
])

null_df.show(truncate=False)

+----------+----+-------+-----+---------+-----------------+-------------------------+-------------------------------+-----------------+------+--------------+---------------+----+------------+-------------+----------+------------------+----------+------------------+------------------+------------------+-----------------+-----------------+---------+-----------------+--------+-----------------+------------------+-----------------+--------+
|FlightDate|Year|Quarter|Month|DayOfWeek|Operating_Airline|Marketing_Airline_Network|Flight_Number_Operating_Airline|Tail_Number      |Origin|OriginCityName|OriginStateName|Dest|DestCityName|DestStateName|CRSDepTime|DepTime           |CRSArrTime|ArrTime           |DepDelay          |DepDel15          |ArrDelay         |ArrDel15         |Cancelled|CancellationCode |Diverted|TaxiOut          |TaxiIn            |AirTime          |Distance|
+----------+----+-------+-----+---------+-----------------+-------------------------+-------------------------------+-

### 11.2 Ranked Percentage View

*Note:* the Spark accumulator-server trace that appears beneath the results below is an internal background-thread warning from PySpark's task-metrics reporting; it is unrelated to the notebook's logic and does not affect the correctness of the results above it.


In [13]:
from pyspark.sql.functions import *
import builtins

rows = eda_df.count()

null_stats = []

for c in eda_df.columns:
    null_count = eda_df.filter(col(c).isNull()).count()
    null_percent = builtins.round((null_count / rows) * 100, 2)

    null_stats.append((c, null_count, null_percent))

null_df = spark.createDataFrame(
    null_stats,
    ["Column", "Null Count", "Null %"]
)

null_df.orderBy(desc("Null %")).show(50, truncate=False)

+-------------------------------+----------+------+
|Column                         |Null Count|Null %|
+-------------------------------+----------+------+
|CancellationCode               |39993169  |97.76 |
|ArrDelay                       |1014879   |2.48  |
|AirTime                        |1014879   |2.48  |
|ArrDel15                       |1014879   |2.48  |
|ArrTime                        |926392    |2.26  |
|TaxiIn                         |926416    |2.26  |
|TaxiOut                        |912686    |2.23  |
|DepTime                        |895495    |2.19  |
|DepDelay                       |896518    |2.19  |
|DepDel15                       |896518    |2.19  |
|Tail_Number                    |296073    |0.72  |
|FlightDate                     |0         |0.0   |
|Quarter                        |0         |0.0   |
|Year                           |0         |0.0   |
|Marketing_Airline_Network      |0         |0.0   |
|Month                          |0         |0.0   |
|DayOfWeek  

### Observation

`CancellationCode` is missing in **97.76%** of records. Delay- and time-related fields show consistent, low single-digit null rates: `ArrDelay`, `ArrDel15`, and `AirTime` at **2.48%**; `ArrTime` and `TaxiIn` at **2.26%**; `TaxiOut`, `DepTime`, `DepDelay`, and `DepDel15` at **2.19%–2.23%**. `Tail_Number` is missing in **0.72%** of rows, and all identifier, schedule, and route columns are 0% missing.

### Business Insight

The pattern of missingness is not random — it is concentrated almost entirely in outcome fields that are undefined for flights that never departed or never landed as scheduled. This means the Silver dataset does not require imputation or row-dropping to "fix" these nulls: the nulls themselves are meaningful signals of cancelled/diverted operations and should be preserved (not imputed) into the Gold layer. This is confirmed formally with operational-status data in Section 12.


## 12. Business Rule Validation — Part 2: Null Composition by Operational Status

Having established in Section 11 that `ArrDelay` and `DepDelay` nulls are concentrated in roughly 2.2%–2.5% of records, this section tests directly whether those nulls are explained by a flight's `Cancelled` or `Diverted` status.


In [14]:
eda_df.filter(col("ArrDelay").isNull()) \
      .groupBy("Cancelled", "Diverted") \
      .count() \
      .show()

+---------+--------+------+
|Cancelled|Diverted| count|
+---------+--------+------+
|      0.0|     1.0| 97790|
|      1.0|     0.0|917084|
|      0.0|     0.0|     5|
+---------+--------+------+

In [15]:
eda_df.filter(col("DepDelay").isNull()) \
      .groupBy("Cancelled", "Diverted") \
      .count() \
      .show()

+---------+--------+------+
|Cancelled|Diverted| count|
+---------+--------+------+
|      1.0|     0.0|896518|
+---------+--------+------+

### Observation

Of the records with a null `ArrDelay`: 917,084 correspond to cancelled (non-diverted) flights and 97,790 correspond to diverted (non-cancelled) flights, with only 5 records falling outside either category. Every record with a null `DepDelay` (896,518 rows) corresponds to a cancelled flight.

### Business Insight

This confirms that `ArrDelay` and `DepDelay` nulls are overwhelmingly **operational business rules, not data-quality defects**: a flight that never departed cannot have a departure delay, and a flight that was cancelled or diverted before reaching its scheduled destination cannot have a standard arrival delay. This finding is extended with a full exception search in Section 21.


## 13. Business Rule Validation — Part 1 Continued: Count-Based Confirmation

The row-level check in Section 9 returned no matching records. The cell below is a retained placeholder comment from the original analysis marking the start of this confirmation step, followed by count-based checks that verify the same rule programmatically.


In [16]:
#Delay rule validation

In [17]:
eda_df.filter(
    (col("ArrDelay") >= 15) &
    (col("ArrDel15") != 1)
).count()

0

In [18]:
eda_df.filter(
    (col("DepDelay") >= 15) &
    (col("DepDel15") != 1)
).count()

0

### Observation

Both the count-based checks return **zero violations**: no record has `ArrDelay >= 15` with `ArrDel15 != 1`, and no record has `DepDelay >= 15` with `DepDel15 != 1`. This matches the empty result sets returned by the row-level check in Section 9.

### Business Insight

This confirms the `ArrDel15`/`DepDel15` flags are 100% reliable derived fields that can be trusted directly for delay-rate KPIs, dashboard filters, and as a clean binary target label for machine learning, without needing to re-derive them from the raw minute values.


## 14. Airline Performance Analysis

Average arrival delay is computed per operating airline to identify which carriers experience the most significant on-time performance challenges.


In [19]:
eda_df.groupBy("Operating_Airline") \
      .agg(avg("ArrDelay").alias("AvgArrivalDelay")) \
      .orderBy(desc("AvgArrivalDelay")) \
      .show(10, False)

+-----------------+------------------+
|Operating_Airline|AvgArrivalDelay   |
+-----------------+------------------+
|G4               |12.900632680867137|
|F9               |12.81508360285754 |
|B6               |12.147421932830499|
|AA               |9.654374415617609 |
|G7               |8.447878313842946 |
|C5               |8.435830927013841 |
|NK               |8.252641895449296 |
|YV               |7.510248904166332 |
|ZW               |6.874314432146032 |
|OH               |6.851562618104394 |
+-----------------+------------------+
only showing top 10 rows

### Observation

Ultra-low-cost and low-cost carriers show the highest average arrival delays: Allegiant (`G4`, 12.90 min), Frontier (`F9`, 12.82 min), and JetBlue (`B6`, 12.15 min) top the list, followed by American Airlines (`AA`, 9.65 min).

### Business Insight

The concentration of high average delays among ultra-low-cost carriers is consistent with industry-wide patterns tied to tighter aircraft turnaround schedules and higher load factors. This ranking is directly usable as an airline scorecard metric in the Power BI dashboard and as a categorical feature (carrier-level historical delay rate) for the ML delay-prediction model.


## 15. Airport Traffic Analysis

Flight volume by origin airport is examined to identify the busiest airports in the network.


In [20]:
eda_df.groupBy("Origin") \
      .count() \
      .orderBy(desc("count")) \
      .show(10, False)

+------+-------+
|Origin|count  |
+------+-------+
|ATL   |1915120|
|ORD   |1778804|
|DFW   |1705528|
|DEN   |1683092|
|CLT   |1347556|
|LAX   |1086362|
|SEA   |1026212|
|PHX   |1021572|
|LAS   |999915 |
|IAH   |904228 |
+------+-------+
only showing top 10 rows

### Observation

Atlanta (`ATL`, 1.92M flights), Chicago O'Hare (`ORD`, 1.78M), and Dallas–Fort Worth (`DFW`, 1.71M) are the busiest origin airports by flight volume, consistent with their roles as major U.S. hub airports.

### Business Insight

Hub-airport volume concentration means delay dynamics at a small number of airports (ATL, ORD, DFW, DEN, CLT) will disproportionately influence system-wide on-time performance. These airports are strong candidates for airport-level congestion features in the Gold layer and should be prioritized for dashboard drill-down views.


## 16. Monthly Trend Analysis

Flight volume is aggregated by month to identify seasonal demand patterns.


In [21]:
eda_df.groupBy("Month") \
      .count() \
      .orderBy("Month") \
      .show()

+-----+-------+
|Month|  count|
+-----+-------+
|    1|3358992|
|    2|3141722|
|    3|3668894|
|    4|3246144|
|    5|3249065|
|    6|3352655|
|    7|3617203|
|    8|3590730|
|    9|3341566|
|   10|3525386|
|   11|3378386|
|   12|3439510|
+-----+-------+

### Observation

Flight volume peaks in March (3.67M) and again in July/August (3.62M / 3.59M), and is lowest in February (3.14M) — a pattern consistent with spring travel, summer peak season, and February's shorter calendar length.

### Business Insight

Seasonal volume swings should inform Gold-layer time features (e.g., a `Season` flag derived from `FlightDate`, planned in Section 25) so the ML model and dashboards can separate genuine seasonal demand effects from carrier- or airport-specific performance issues.


## 17. Correlation Analysis

Pearson correlation is computed between key numeric pairs to validate internal consistency of the dataset and to identify which features carry redundant or strongly related information.


In [22]:
print("DepDelay vs ArrDelay:", eda_df.stat.corr("DepDelay","ArrDelay"))
print("Distance vs AirTime:", eda_df.stat.corr("Distance","AirTime"))

DepDelay vs ArrDelay: 0.9626657385544707
Distance vs AirTime: 0.9467148729106268

### Observation

A strong positive correlation exists between **Departure Delay and Arrival Delay** (r = 0.963), confirming that delays typically originate at departure and propagate through to arrival rather than being independently generated en route. **Distance and Air Time** are also strongly correlated (r = 0.947), as expected physically.

### Business Insight

Because departure delay so strongly predicts arrival delay, departure-time features (`DepDelay`, and `CRSDepTime`-derived peak-hour flags) are likely to be among the most powerful predictors in the Gold-layer machine learning delay model, since recovery time in the air is limited. The Distance–AirTime correlation confirms the dataset's flight-duration measurements are internally consistent and reliable for efficiency-based operational metrics.


## 18. Composite Business Key Validation — Part 1: Five-Column Candidate

No single column in this dataset uniquely identifies a flight record — the same flight number can be reused by an airline on different dates, and the same aircraft or route can appear multiple times per day. This section tests an initial five-column candidate key.


In [23]:
candidate = [
    "FlightDate",
    "Flight_Number_Operating_Airline",
    "Origin",
    "Dest",
    "CRSDepTime"
]

print("Rows:", eda_df.count())
print("Unique:", eda_df.select(*candidate).dropDuplicates().count())

Rows: 40910253
Unique: 40910001

### Observation

The five-column candidate (`FlightDate`, `Flight_Number_Operating_Airline`, `Origin`, `Dest`, `CRSDepTime`) is **not** a valid key: 40,910,253 total rows produce only 40,910,001 unique combinations — a shortfall of 252 rows. This is explained by `Flight_Number_Operating_Airline` differing from the number under which the airline markets the ticket (e.g., on codeshare flights), so two distinct marketed flights can collide on the same operating flight number, date, route, and scheduled departure time.

### Business Insight

Using an incomplete key in the Gold layer would risk silently merging or double-counting distinct flights during joins and aggregations — a serious data-integrity risk for KPI accuracy. This finding directly motivated testing an expanded, marketing-airline-aware key, carried out in Section 20.


In [24]:
#NEW COLUMNS

In [25]:
selected_cols = [
    "FlightDate","Year","Quarter","Month","DayOfMonth","DayOfWeek",
    "Marketing_Airline_Network","Operating_Airline",
    "Origin","Dest",
    "CRSDepTime","CRSArrTime",
    "DepDelay","ArrDelay","DepDel15","ArrDel15",
    "CarrierDelay","WeatherDelay","NASDelay",
    "SecurityDelay","LateAircraftDelay",
    "Cancelled","CancellationCode","Diverted",
    "Distance","AirTime","TaxiOut","TaxiIn"
]

selected_cols = [c for c in selected_cols if c in df.columns]

eda_df = df.select(*selected_cols)

> **Note:** From this point forward, `eda_df` is redefined to reference a refined 28-column feature set (`selected_cols`) that adds the delay-cause columns (`CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay`) required for the remaining validation and feature-selection work. See Section 25 for the complete column listing and business rationale.


## 19. Delay Cause Analysis

The refined feature set includes the five DOT-standard delay-cause columns. Their distribution and null pattern are examined to understand which factors drive delays and to confirm when cause attribution is (and is not) expected to be present.

### 19.1 Delay Cause Summary Statistics


In [26]:
delay_causes = [
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay"
]

eda_df.select(delay_causes).describe().show()

+-------+------------------+-----------------+------------------+-------------------+------------------+
|summary|      CarrierDelay|     WeatherDelay|          NASDelay|      SecurityDelay| LateAircraftDelay|
+-------+------------------+-----------------+------------------+-------------------+------------------+
|  count|           7644267|          7644267|           7644267|            7644267|           7644267|
|   mean|25.263957028188575|4.279215521906809|13.147254144838216|0.13603436405347955| 27.12500452953828|
| stddev| 75.45408450244801|34.04622253656452|31.962116802658198| 3.5225089292055705|61.031506334288096|
|    min|               0.0|              0.0|               0.0|                0.0|               0.0|
|    max|            7232.0|           2419.0|            2700.0|             1460.0|            3581.0|
+-------+------------------+-----------------+------------------+-------------------+------------------+

### Observation

Delay-cause columns are populated for 7,644,267 records — closely matching the 7,644,272 flights flagged as delayed (`ArrDel15 = 1`) in Section 8. Among delayed flights, **Late Aircraft Delay** (mean 27.1 min) and **Carrier Delay** (mean 25.3 min) are the largest average contributors, followed by **NAS (National Airspace System) Delay** (13.1 min), while **Weather Delay** (4.3 min) and **Security Delay** (0.14 min) are comparatively minor.

### Business Insight

Late Aircraft Delay being the single largest contributor reinforces the finding from Section 17 that delays cascade through the day's schedule — an aircraft arriving late from a prior leg is a leading cause of subsequent delay. This means airlines can gain more from improving turnaround and recovery buffer time than from weather mitigation alone, which is valuable context for a Gold-layer root-cause dashboard.

### 19.2 Delay Cause Null Pattern


In [27]:
for c in delay_causes:
    print(c)
    eda_df.filter(col(c).isNull()).count()

CarrierDelay
33265986
WeatherDelay
33265986
NASDelay
33265986
SecurityDelay
33265986
LateAircraftDelay
33265986

### Observation

All five delay-cause columns share an identical null count of 33,265,986 — exactly equal to the number of flights that were **not** flagged as delayed (40,910,253 − 7,644,267). This confirms delay-cause attribution is only recorded for flights that were actually delayed, per DOT convention.

### Business Insight

Because the five cause columns are null (not zero) for on-time flights, any aggregation (e.g., "average weather delay across all flights") must explicitly treat nulls as "not applicable" rather than "zero delay," to avoid understating the severity of delay causes among the flights they actually affect.


## 20. Composite Business Key Validation — Part 2: Final Six-Column Key

The final candidate key extends the five-column version from Section 18 with `Marketing_Airline_Network` and switches to `Flight_Number_Marketing_Airline`, distinguishing flights by the airline that sold the ticket rather than only the airline that operated it. Column availability is confirmed first, followed by a full uniqueness check.


In [28]:
candidate_key = [
    "FlightDate",
    "Marketing_Airline_Network",
    "Flight_Number_Marketing_Airline",
    "Origin",
    "Dest",
    "CRSDepTime"
]

missing = [c for c in candidate_key if c not in df.columns]

if missing:
    print("Missing Columns:", missing)
else:
    print("All columns are available.")

All columns are available.

In [29]:
candidate_key = [
    "FlightDate",
    "Marketing_Airline_Network",
    "Flight_Number_Marketing_Airline",
    "Origin",
    "Dest",
    "CRSDepTime"
]

total_rows = df.count()

unique_rows = (
    df.select(*candidate_key)
      .dropDuplicates()
      .count()
)

print("Total Rows :", total_rows)
print("Unique Rows:", unique_rows)
print("Duplicates :", total_rows - unique_rows)

Total Rows : 40910253
Unique Rows: 40910253
Duplicates : 0

### Composite Business Key — Conclusion

The six-column combination:

- `FlightDate`
- `Marketing_Airline_Network`
- `Flight_Number_Marketing_Airline`
- `Origin`
- `Dest`
- `CRSDepTime`

uniquely identifies every record in the Silver dataset: **40,910,253 total rows and 40,910,253 unique combinations — zero duplicates**, resolving the 252-row collision found with the five-column candidate in Section 18.

**Why a composite key is required:** no single column can serve as a primary key for this dataset.
- `FlightDate` alone repeats across thousands of flights per day.
- `Flight_Number_Marketing_Airline` alone repeats across different dates and different operating carriers under codeshare agreements.
- `Origin` / `Dest` alone repeat across every flight on that route.
- `CRSDepTime` alone repeats across flights departing at the same scheduled time from different airports.

Each column is necessary but not individually sufficient: together, the date, the marketing airline, the flight number, the route (origin and destination), and the scheduled departure time jointly pin down exactly one flight record. `Marketing_Airline_Network` and `Flight_Number_Marketing_Airline` are specifically required — rather than the operating-carrier equivalents used in Section 18 — because they reflect how the flight is uniquely sold and numbered, avoiding the operating-flight-number collisions identified there.

### Business Insight

This composite key is the foundation for all downstream joins, deduplication checks, and slowly-changing-dimension logic in the Gold layer. Using it consistently prevents accidental record duplication or loss when merging Silver-layer data with reference tables (airports, airlines) or historical Gold-layer snapshots.


## 21. Business Rule Validation — Detailed Null-Composition Walkthrough

This section provides an independent, step-by-step re-verification of the relationship between missing `ArrDelay` values and flight operational status, extending the Section 12 check with an explicit search for exception records.

### Step 1: Count Cancelled and Diverted Flights


In [31]:
from pyspark.sql.functions import col, sum

eda_df.select(
    sum(col("Cancelled")).alias("Cancelled_Flights"),
    sum(col("Diverted")).alias("Diverted_Flights")
).show()

+-----------------+----------------+
|Cancelled_Flights|Diverted_Flights|
+-----------------+----------------+
|         917084.0|         97790.0|
+-----------------+----------------+

### Step 2: Count Rows Where ArrDelay Is Null


In [32]:
from pyspark.sql.functions import col

arrdelay_nulls = eda_df.filter(col("ArrDelay").isNull()).count()
print("ArrDelay NULLs:", arrdelay_nulls)

ArrDelay NULLs: 1014879

### Step 3: Composition of ArrDelay Nulls by Operational Status


In [34]:
eda_df.filter(col("ArrDelay").isNull()) \
      .groupBy("Cancelled", "Diverted") \
      .count() \
      .show()

+---------+--------+------+
|Cancelled|Diverted| count|
+---------+--------+------+
|      0.0|     1.0| 97790|
|      1.0|     0.0|917084|
|      0.0|     0.0|     5|
+---------+--------+------+

### Step 4: Identify Any Exceptions


In [35]:
eda_df.filter(
    (col("ArrDelay").isNull()) &
    (col("Cancelled") == 0) &
    (col("Diverted") == 0)
).show(20, truncate=False)

+-------------------+----+-------+-----+---------+-------------------------+-----------------+------+----+----------+----------+--------+--------+--------+--------+------------+------------+--------+-------------+-----------------+---------+----------------+--------+--------+-------+-------+------+
|FlightDate         |Year|Quarter|Month|DayOfWeek|Marketing_Airline_Network|Operating_Airline|Origin|Dest|CRSDepTime|CRSArrTime|DepDelay|ArrDelay|DepDel15|ArrDel15|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|Cancelled|CancellationCode|Diverted|Distance|AirTime|TaxiOut|TaxiIn|
+-------------------+----+-------+-----+---------+-------------------------+-----------------+------+----+----------+----------+--------+--------+--------+--------+------------+------------+--------+-------------+-----------------+---------+----------------+--------+--------+-------+-------+------+
|2025-09-06 00:00:00|2025|3      |9    |6        |DL                       |YX               |JFK   

### Observation

This independent walkthrough reproduces the Section 12 finding exactly: of the 1,014,879 null `ArrDelay` records, 917,084 are cancelled flights and 97,790 are diverted (non-cancelled) flights, together accounting for 1,014,874 of the 1,014,879 nulls (99.9995%). Step 4 isolates the remaining **5 exception records** — flights that are neither cancelled nor diverted but still have a null `ArrDelay`. All five involve short-haul or regional routes (JFK–MVY, DTW–IND, BOS–CLE, DEN–LAS, LGA–BNA) with a valid `DepDelay` but a missing arrival record, suggesting isolated data-capture gaps (e.g., an unrecorded landing time) rather than a systemic issue.

### Business Insight

With only 5 unexplained records out of 40.9 million (0.00001%), the missing-value pattern in `ArrDelay` is overwhelmingly explained by legitimate operational status rather than data-pipeline defects — this is strong evidence the Silver layer is Gold-ready without additional cleansing, aside from optionally flagging these 5 records for manual review.


## 22. Note on Retained Placeholder Cell

The following empty cell was retained from the original analysis notebook. It appears to have been an early placeholder for the monthly flight-volume check that was ultimately implemented in Section 16 (Monthly Trend Analysis). It is preserved here, unmodified, for completeness and audit traceability.


## 23. Full Raw Schema Reference

Before finalizing feature selection, the complete raw Silver-layer schema (all 120 columns) is reviewed. This full inventory is the basis for the Excluded Features justification in Section 25.


In [30]:
print("Rows :", df.count())
print("Columns :", len(df.columns))

df.printSchema()

Rows : 40910253
Columns : 120
root
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: timestamp (nullable = true)
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- Operated_or_Branded_Code_Share_Partners: string (nullable = true)
 |-- DOT_ID_Marketing_Airline: integer (nullable = true)
 |-- IATA_Code_Marketing_Airline: string (nullable = true)
 |-- Flight_Number_Marketing_Airline: integer (nullable = true)
 |-- Originally_Scheduled_Code_Share_Airline: string (nullable = true)
 |-- DOT_ID_Originally_Scheduled_Code_Share_Airline: integer (nullable = true)
 |-- IATA_Code_Originally_Scheduled_Code_Share_Airline: string (nullable = true)
 |-- Flight_Num_Originally_Scheduled_Code_Share_Airline: integer (nullable = true)
 |-- Operating_Airline: string (nullable = true)
 |-- DOT_ID_Operating_Airline: integer (nullable = true)
 |-- IATA_Code_Operating_Airl

### Observation

The raw Silver dataset contains **40,910,253 rows and 120 columns** — roughly four times the ~30 columns used in this EDA. Categories of columns not carried into the working `eda_df` include: verbose identifier duplicates (e.g., `DOT_ID_Marketing_Airline`, `IATA_Code_Marketing_Airline`), redundant descriptive fields (`OriginCityName`, `OriginStateName`, `DestCityName`, `DestStateName` alongside the coded `Origin`/`Dest`), granular operational timestamps (`WheelsOff`, `WheelsOn`, `FirstDepTime`), redundant elapsed-time measures (`CRSElapsedTime`, `ActualElapsedTime`, `DistanceGroup`), and an extensive block of diversion-detail columns (`Div1Airport` through `Div5TailNum`) populated only for the small fraction of diverted flights. Two raw artifact columns (`Duplicate`, `_c119`) also appear, consistent with residual source-file formatting from the original DOT on-time-performance CSV extract.

### Business Insight

Confirming the full 120-column inventory validates that the reduced EDA and feature subsets (Sections 3 and 25) were deliberate, business-driven simplifications rather than accidental omissions — an important point to document for anyone auditing the Gold-layer feature-engineering decisions.

> The following empty cell was also retained from the original notebook and is preserved unmodified for completeness.


## 24. EDA Summary

### Dataset Quality

The Silver-layer dataset contains **40,910,253 flight records across 120 raw columns**, of which a focused subset of business-critical columns was selected for analysis (Sections 3 and 25). The dataset is well-typed, consistently formatted, and free of structural loading issues.

### Missing Values

Missing values are concentrated in a small number of outcome-dependent fields: `CancellationCode` (97.76%, expected — only populated for cancelled flights), and a cluster of delay/timing fields (`ArrDelay`, `ArrDel15`, `AirTime`, `ArrTime`, `TaxiIn`, `TaxiOut`, `DepTime`, `DepDelay`, `DepDel15`) all missing in the 2.19%–2.48% range. `Tail_Number` is missing in 0.72% of rows. Business Rule Validation (Sections 12 and 21) confirmed that these delay-field nulls are almost entirely (99.9995%) explained by cancelled and diverted flights, with only 5 unexplained exception records across the entire dataset.

### Correlation Findings

Departure Delay and Arrival Delay are strongly positively correlated (r = 0.963), confirming that delays predominantly originate at departure and propagate forward. Distance and Air Time are also strongly correlated (r = 0.947), confirming internally consistent flight-duration measurements.

### Delay Trends

Approximately **18.7% of flights arrive 15+ minutes late** and **18.5% depart 15+ minutes late**, with a **2.24% cancellation rate** and **0.24% diversion rate**. Among delayed flights, Late Aircraft Delay (27.1 min average) and Carrier Delay (25.3 min average) are the largest contributing causes, ahead of NAS Delay (13.1 min), Weather Delay (4.3 min), and Security Delay (0.14 min).

### Airline Insights

Ultra-low-cost and low-cost carriers (Allegiant, Frontier, JetBlue) show the highest average arrival delays, while American Airlines' marketed-flight volume (10.4M) substantially exceeds its operated-flight volume (5.1M) due to codeshare/regional-partner operations — a distinction that must be preserved in any airline performance scoring.

### Airport Insights

Atlanta (ATL), Chicago O'Hare (ORD), and Dallas–Fort Worth (DFW) are the busiest origin airports, consistent with their status as major U.S. hubs. Flight volume peaks in March and mid-summer and dips in February, indicating clear seasonal demand patterns.

### Business Rule Validation

Two independent rule checks confirmed data integrity: (1) the `ArrDel15`/`DepDel15` flags perfectly match the 15-minute delay threshold applied to `ArrDelay`/`DepDelay`, with zero violations found across both a row-level and count-based check; (2) missing delay values are explained by cancellation/diversion status in 99.9995% of cases, with only 5 unexplained exception records.

### Composite Key Validation

A five-column candidate key (`FlightDate`, `Flight_Number_Operating_Airline`, `Origin`, `Dest`, `CRSDepTime`) was found insufficient (252 duplicate collisions). Replacing the operating-flight identifiers with `Marketing_Airline_Network` and `Flight_Number_Marketing_Airline` produced a fully valid six-column composite business key with zero duplicates across all 40,910,253 records.

---


## 25. Data Understanding & Feature Selection

### Objective

The objective of this section is to formally justify the business-critical features selected during this EDA for use in Gold-layer feature engineering, KPI generation, machine learning model development, and Power BI dashboard construction. Feature selection reduces the raw 120-column Silver schema to a compact, business-relevant set, improving Spark processing efficiency, model interpretability, and dashboard performance while avoiding redundant or low-value columns.

### Dataset Understanding

The Silver-layer dataset covers U.S. domestic flight operations from **2020–2025**, with **40,910,253 records** and **120 raw columns** sourced from the DOT on-time-performance data. It captures flight identity (airline, flight number, tail number), scheduling (dates, scheduled/actual times), routing (origin/destination airports and cities), delay outcomes (departure and arrival delay in minutes, 15-minute delay flags), delay causes (carrier, weather, NAS, security, late aircraft), operational status (cancelled, diverted, and diversion detail), and flight metrics (distance, air time, taxi times). This EDA confirmed the dataset is complete for its core identifier and schedule fields, with missingness confined to outcome fields that are conditionally undefined for cancelled or diverted flights.

### Column Categorization

The 28 columns carried forward into the refined `eda_df` (Section 18) are grouped below by business category.

| Category | Columns |
|---|---|
| Time Features | FlightDate, Year, Quarter, Month, DayOfMonth, DayOfWeek |
| Airline Features | Marketing_Airline_Network, Operating_Airline |
| Airport / Route Features | Origin, Dest |
| Schedule Features | CRSDepTime, CRSArrTime |
| Delay Features | DepDelay, ArrDelay, DepDel15, ArrDel15 |
| Delay Cause Features | CarrierDelay, WeatherDelay, NASDelay, SecurityDelay, LateAircraftDelay |
| Operational Features | Cancelled, CancellationCode, Diverted |
| Flight Metrics | Distance, AirTime, TaxiOut, TaxiIn |

### Selected Features

| Column | Category | Business Purpose | Used in Gold Layer | Used for ML | Used in Dashboard |
|---|---|---|---|---|---|
| FlightDate | Time | Base date for seasonality, weekday, and trend features | Yes | Yes | Yes |
| Year | Time | Multi-year trend comparison | Yes | Yes | Yes |
| Quarter | Time | Quarterly performance reporting | Yes | Yes | Yes |
| Month | Time | Monthly / seasonal trend analysis | Yes | Yes | Yes |
| DayOfMonth | Time | Day-of-month / pay-cycle travel pattern analysis | Yes | Partial | Partial |
| DayOfWeek | Time | Weekday vs. weekend travel pattern analysis | Yes | Yes | Yes |
| Marketing_Airline_Network | Airline | Ticket-selling airline; part of composite key and airline KPIs | Yes | Yes | Yes |
| Operating_Airline | Airline | Actual operating carrier; drives airline performance scoring | Yes | Yes | Yes |
| Origin | Route | Departure airport; core routing and congestion dimension | Yes | Yes | Yes |
| Dest | Route | Arrival airport; core routing dimension | Yes | Yes | Yes |
| CRSDepTime | Schedule | Scheduled departure time; source for peak-hour features | Yes | Yes | Yes |
| CRSArrTime | Schedule | Scheduled arrival time; schedule-adherence analysis | Yes | Yes | Partial |
| DepDelay | Delay | Departure delay in minutes; primary delay driver | Yes | Yes | Yes |
| ArrDelay | Delay | Arrival delay in minutes; primary target metric | Yes | Yes | Yes |
| DepDel15 | Delay | Binary departure-delay flag; validated business rule | Yes | Yes | Yes |
| ArrDel15 | Delay | Binary arrival-delay flag; primary ML target label | Yes | Yes | Yes |
| CarrierDelay | Delay Cause | Root-cause attribution: airline-controllable delay | Yes | Yes | Yes |
| WeatherDelay | Delay Cause | Root-cause attribution: weather-driven delay | Yes | Yes | Yes |
| NASDelay | Delay Cause | Root-cause attribution: air-traffic-system delay | Yes | Yes | Yes |
| SecurityDelay | Delay Cause | Root-cause attribution: security-driven delay | Yes | Partial | Yes |
| LateAircraftDelay | Delay Cause | Root-cause attribution: inbound-aircraft cascade delay | Yes | Yes | Yes |
| Cancelled | Operational | Cancellation flag; reliability scoring | Yes | Yes | Yes |
| CancellationCode | Operational | Cancellation reason; dashboard root-cause breakdown | Yes | Partial | Yes |
| Diverted | Operational | Diversion flag; reliability scoring | Yes | Yes | Yes |
| Distance | Flight Metrics | Route length; efficiency and category features | Yes | Yes | Yes |
| AirTime | Flight Metrics | Actual flight duration; efficiency analysis | Yes | Yes | Partial |
| TaxiOut | Flight Metrics | Ground-departure efficiency metric | Yes | Yes | Partial |
| TaxiIn | Flight Metrics | Ground-arrival efficiency metric | Yes | Partial | Partial |

### Excluded Features

| Excluded Column(s) | Reason for Exclusion |
|---|---|
| Tail_Number | High-cardinality aircraft identifier with limited standalone predictive value for route/airline-level analytics; retained only for the initial 30-column EDA scope in Section 3 |
| OriginCityName, OriginStateName, DestCityName, DestStateName | Redundant with the coded `Origin`/`Dest` airport identifiers; useful for display labels only, not for modelling or joins |
| DepTime, ArrTime (actual clock times) | Redundant once `CRSDepTime`/`CRSArrTime` and `DepDelay`/`ArrDelay` are retained; actual time can be reconstructed if ever needed |
| CRSElapsedTime, ActualElapsedTime, DistanceGroup | Redundant with `Distance` and `AirTime`, which already capture flight-duration and length information |
| WheelsOff, WheelsOn, FirstDepTime, TotalAddGTime, LongestAddGTime | Granular ground-operations timestamps not required for delay- and reliability-focused analytics |
| Div1Airport … Div5TailNum (diversion detail block) | Populated only for the small fraction of diverted flights (~0.24%); excessive granularity for Gold-layer KPI and ML scope |
| DOT_ID_*, IATA_Code_* identifier duplicates | Redundant airline/airport identifier codes already represented by `Operating_Airline`/`Marketing_Airline_Network`, `Origin`, and `Dest` |
| Duplicate, _c119 | Residual raw-file artifact columns from the original source extract; not business data |

### Transformation Decisions

| Feature | Planned Transformation | Reason |
|---|---|---|
| FlightDate | → Season | Captures seasonal demand patterns identified in Section 16 |
| FlightDate | → Weekend | Captures weekday vs. weekend travel behaviour |
| CRSDepTime | → DepartureHour | Simplifies scheduled time into an analyzable hourly bucket |
| CRSDepTime | → PeakHour | Flags high-congestion departure windows for delay prediction |
| Origin + Dest | → Route | Creates a single route-level dimension for traffic and reliability analysis |
| Distance | → DistanceCategory | Buckets flights into short/medium/long-haul segments for comparative analysis |
| ArrDelay | → DelaySeverity | Converts continuous delay minutes into business-friendly severity tiers |
| Cancelled | → Reliability metrics | Feeds airline/airport-level on-time and completion-rate KPIs |

### Conclusion

The features selected in this EDA — time, airline, route, schedule, delay, delay-cause, operational, and flight-metric columns — provide the minimum sufficient set of business-relevant attributes required to:

- **Gold Layer:** engineer derived features (season, peak hour, route, distance category, delay severity) and enforce the validated six-column composite business key for reliable joins and deduplication.
- **Machine Learning:** train a delay-prediction model using `ArrDel15`/`ArrDelay` as the target, with departure-time, airline, route, and delay-cause features as leading predictors, informed by the strong departure-arrival delay correlation found in Section 17.
- **Dashboard Development:** power Power BI visuals for airline scorecards, airport congestion drill-downs, seasonal trend charts, and delay root-cause breakdowns, all backed by fields verified for completeness and business-rule consistency in this notebook.

Because business-rule validation showed that the vast majority of missing values are expected operational outcomes rather than data-quality defects, the Gold layer can focus its effort on feature engineering and enrichment rather than extensive data cleansing.
